In [1]:
#  ----------------------------
## opt_pose_calib_sim3函数测试
#  --------------------------- 

import torch
import lietorch
from mast3r_slam.tracker import FrameTracker
from mast3r_slam.frame import Frame
from mast3r_slam.config import load_config
from mast3r_slam.geometry import project_calib

load_config('config/base.yaml')

def make_frame(idx):
    img = torch.zeros(1, 3, 8, 8)
    img_shape = torch.tensor([[8, 8]])
    img_true_shape = img_shape.clone()
    uimg = torch.zeros(8, 8, 3)
    frame = Frame(idx, img, img_shape, img_true_shape, uimg, lietorch.Sim3.Identity(1))
    H = 4
    pts = torch.randn(H * H, 3)
    pts[:, 2] = pts[:, 2].abs() + 1.0
    frame.X_canon = pts
    frame.C = torch.ones(H * H, 1)
    frame.Sigma = torch.diag(torch.tensor([0.02, 0.02, 0.08]) ** 2)
    frame.K = torch.eye(3)
    return frame

kf = make_frame(0)
frame = make_frame(1)

tracker = FrameTracker(model=None, frames=None, device='cpu')
tracker.keyframes = type('KFStore', (), {'last_keyframe': lambda self: kf})()

N = kf.X_canon.shape[0]
Xf = frame.X_canon
Xk = kf.X_canon
Qk = torch.ones(N, 1)
valid = torch.ones(N, 1, dtype=torch.bool)
K = torch.eye(3)
img_size = (4, 4)
meas_k, _, valid_proj = project_calib(Xk, K, img_size, jacobian=True)
valid_meas_k = valid_proj.to(torch.bool)

print('---- Input Samples ----')
print('Xf[0:3]:\n', Xf[:3])
print('Sigma_k:\n', kf.Sigma)
print('Sigma_f:\n', frame.Sigma)
print('Valid mask sum:', valid.sum().item())

T_WCf, T_CkCf = tracker.opt_pose_calib_sim3(
    Xf,
    Xk,
    lietorch.Sim3.Identity(1),
    lietorch.Sim3.Identity(1),
    Qk,
    valid,
    meas_k,
    valid_meas_k,
    K,
    img_size,
    kf.Sigma,
    frame.Sigma,
)

print('\n---- Output ----')
print('T_WCf (Sim3 data):', T_WCf.data)
print('T_CkCf (Sim3 data):', T_CkCf.data)

/home/zhuke/miniconda3/envs/mast3r/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---- Input Samples ----
Xf[0:3]:
 tensor([[ 0.5106, -0.3099,  1.3269],
        [-0.3796, -0.2335,  1.4750],
        [ 0.3152, -0.4551,  3.1009]])
Sigma_k:
 tensor([[0.0004, 0.0000, 0.0000],
        [0.0000, 0.0004, 0.0000],
        [0.0000, 0.0000, 0.0064]])
Sigma_f:
 tensor([[0.0004, 0.0000, 0.0000],
        [0.0000, 0.0004, 0.0000],
        [0.0000, 0.0000, 0.0064]])
Valid mask sum: 16

---- Output ----
T_WCf (Sim3 data): tensor([[-0.3177,  0.0585,  1.1128, -0.1568,  0.1205, -0.0330,  0.9797,  0.5009]])
T_CkCf (Sim3 data): tensor([[-0.3177,  0.0585,  1.1128, -0.1568,  0.1205, -0.0330,  0.9797,  0.5009]])


In [1]:
#  ----------------------------
## update_pointmap 协方差融合测试
#  ----------------------------

import torch
import lietorch
from mast3r_slam.frame import Frame
from mast3r_slam.geometry import sim3_point_linear, propagate_covariance
from mast3r_slam.config import load_config

load_config('config/base.yaml')


def make_frame(idx):
    img = torch.zeros(1, 3, 8, 8)
    img_shape = torch.tensor([[8, 8]])
    img_true_shape = img_shape.clone()
    uimg = torch.zeros(8, 8, 3)
    frame = Frame(idx, img, img_shape, img_true_shape, uimg, lietorch.Sim3.Identity(1))
    H = 4
    pts = torch.randn(H * H, 3)
    pts[:, 2] = pts[:, 2].abs() + 1.0
    frame.X_canon = pts
    frame.C = torch.ones(H * H, 1)
    frame.Sigma = torch.diag(torch.tensor([0.02, 0.02, 0.08]) ** 2)
    return frame

kf = make_frame(0)
new_frame = make_frame(1)

print('Before fusion:')
print('kf Sigma =\n', kf.Sigma)
print('kf first pts =\n', kf.X_canon[:2])

A = sim3_point_linear(lietorch.Sim3.Identity(1))
Sigma_rot = propagate_covariance(new_frame.Sigma, A)

kf.update_pointmap(new_frame.X_canon, new_frame.C, Sigma_rot)

print('\nAfter fusion:')
print('kf Sigma =\n', kf.Sigma)
print('kf first pts =\n', kf.X_canon[:2])


/home/zhuke/miniconda3/envs/mast3r/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Before fusion:
kf Sigma =
 tensor([[0.0004, 0.0000, 0.0000],
        [0.0000, 0.0004, 0.0000],
        [0.0000, 0.0000, 0.0064]])
kf first pts =
 tensor([[ 1.4918,  1.5538,  1.9143],
        [ 0.1924, -1.5596,  3.0864]])

After fusion:
kf Sigma =
 tensor([[0.0004, 0.0000, 0.0000],
        [0.0000, 0.0004, 0.0000],
        [0.0000, 0.0000, 0.0064]])
kf first pts =
 tensor([[1.8281, 0.7680, 2.6369],
        [0.9091, 0.9704, 1.0262]])
